In [2]:
import pandas as pd

final_data = "../results/final_housing_data.csv"
df = pd.read_csv(final_data)

In [35]:
# calculate summary statistics for the dataset
summary_stats = df.describe()
print(summary_stats)

       housing_concern_count  total_students  percent_housing_concern
count              38.000000       38.000000                38.000000
mean               33.500000      579.210526                 6.655572
std                20.448683      438.027098                 2.199565
min                 4.000000       83.000000                 3.736264
25%                17.000000      232.500000                 5.030931
50%                30.000000      479.000000                 5.868757
75%                47.250000      856.000000                 8.444816
max                82.000000     1947.000000                11.627907


In [36]:
# QC: calculate the total district percent for housing concern
# result: manual check indicates matching value with the district-level percent in the Atlanta Pulic Schools.xls
hcc_sum = df["housing_concern_count"].sum()
ts_sum = df["total_students"].sum()

total_district_value = hcc_sum/ts_sum * 100
print(total_district_value)

5.783734666060882


In [37]:
# top 5 schools with the highest percent of students with housing concern
top5_highest = df.nlargest(5, "percent_housing_concern")[["school", "housing_concern_count", "total_students", "percent_housing_concern"]]
display(top5_highest)

# top 5 schools with the lowest percent of students with housing concern
top5_lowest = df.nsmallest(5, "percent_housing_concern")[["school", "housing_concern_count", "total_students", "percent_housing_concern"]]
display(top5_lowest)

,school,housing_concern_count,total_students,percent_housing_concern
27,Michael R. Hollis Innovation Academy,20,172,11.627907
15,Hillside Conant School,9,83,10.843373
18,Judson Price Middle School,24,227,10.572687
33,The Kindezi School,12,115,10.434783
7,Centennial Place Academy (Charter),21,202,10.396040


,school,housing_concern_count,total_students,percent_housing_concern
5,Carver High School,17,455,3.736264
29,North Atlanta High School,82,1947,4.211608
22,Kindezi Old 4th Ward,4,93,4.301075
30,Ralph Bunche Middle School,35,791,4.424779
28,Midtown High School,64,1424,4.494382


### Create map of housing concern percentages

In [39]:
# extract school coordinates for mapping

locations = "data/EDGE_GEOCODE_PUBLICSCH_2425.xlsx"
loc_df = pd.read_excel(locations, usecols=["NAME", "CITY", "STATE", "LAT", "LON"])
display(loc_df.head())

,NAME,CITY,STATE,LAT,LON
0,Albertville Middle School,Albertville,AL,34.260200,-86.206200
1,Albertville High School,Albertville,AL,34.261772,-86.204911
2,Albertville Intermediate School,Albertville,AL,34.273557,-86.220154
3,Albertville Elementary School,Albertville,AL,34.252700,-86.221806
4,Albertville Kindergarten and PreK,Albertville,AL,34.289945,-86.193056


In [45]:
locations_filtered = loc_df[
    (loc_df["NAME"].isin(df["school"])) &
    (loc_df["STATE"] == "GA") &
    (loc_df["CITY"] == "Atlanta")
]

locations_filtered = locations_filtered.sort_values("NAME").reset_index(drop=True)
display(locations_filtered)

locations_filtered.to_csv("../results/school_locations.csv", index=False)

,NAME,CITY,STATE,LAT,LON
0,Atlanta Classical Academy,Atlanta,GA,33.843780,-84.407711
1,Atlanta Neighborhood Charter - Middle,Atlanta,GA,33.732522,-84.355146
2,B.E.S.T Academy,Atlanta,GA,33.788712,-84.479439
3,Benjamin E. Mays High School,Atlanta,GA,33.733900,-84.503800
4,Booker T. Washington High School,Atlanta,GA,33.753637,-84.420896
5,Carver High School,Atlanta,GA,33.719134,-84.389060
6,Carver High School Early College,Atlanta,GA,33.719100,-84.388800
7,Centennial Place Academy (Charter),Atlanta,GA,33.769700,-84.395400
8,Coretta Scott King Young Women's Leadership Ac...,Atlanta,GA,33.789300,-84.479300
9,Crawford Long Middle School,Atlanta,GA,33.666500,-84.394400


In [59]:
# merge location data with main dataset

school_loc_data = "../results/school_locations.csv"
school_loc = pd.read_csv(school_loc_data)

df_with_loc = pd.merge(df, school_loc, left_on="school", right_on="NAME", how="left")
df_with_loc = df_with_loc.drop(columns=["file", "NAME", "CITY", "STATE"])
df_with_loc = df_with_loc.rename(columns={"LAT": "lat", "LON": "lon"})
display(df_with_loc)

df_with_loc.to_csv("../results/final_housing_data_with_loc.csv", index=False)

,school,housing_concern_count,total_students,percent_housing_concern,lat,lon
0,Atlanta Classical Academy,36,748,4.812834,33.843780,-84.407711
1,Atlanta Neighborhood Charter - Middle,45,504,8.928571,33.732522,-84.355146
2,B.E.S.T Academy,21,282,7.446809,33.788712,-84.479439
3,Benjamin E. Mays High School,42,885,4.745763,33.733900,-84.503800
4,Booker T. Washington High School,61,793,7.692308,33.753637,-84.420896
5,Carver High School,17,455,3.736264,33.719134,-84.389060
6,Carver High School Early College,23,503,4.572565,33.719100,-84.388800
7,Centennial Place Academy (Charter),21,202,10.396040,33.769700,-84.395400
8,Coretta Scott King Young Women's Leadership Ac...,33,346,9.537572,33.789300,-84.479300
9,Crawford Long Middle School,31,608,5.098684,33.666500,-84.394400


In [8]:
# map of percent of students reporting stress from housing concerns by school location
import folium
from branca.colormap import LinearColormap

data_with_loc = pd.read_csv("../results/final_housing_data_with_loc.csv")

m = folium.Map(
    location=[data_with_loc["lat"].mean(), data_with_loc["lon"].mean()],
    zoom_start=11
)

colormap = LinearColormap(
    colors=["green", "yellow", "red"],
    vmin=data_with_loc["percent_housing_concern"].min(),
    vmax=data_with_loc["percent_housing_concern"].max()
)

colormap.caption = "Percent of Students Reporting Stress From Housing Concerns"

for _, row in data_with_loc.iterrows():

    color = colormap(row["percent_housing_concern"])

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        color="black",
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=f"{row['school']}<br>{row['percent_housing_concern']:.1f}%"
    ).add_to(m)

colormap.add_to(m)

m